<a href="https://colab.research.google.com/github/jdyar423-gif/TuringAward2027/blob/claude%2Fmodest-carson-yi2d9g/notebooks/colab_v6e_24h_v9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BES-100M-A20M: the 24-hour run on one TPU v6e, configuration v9

This notebook runs `scripts/run_24h.py` with `configs/v9.py` (a page of private neurons and
output gains per vocabulary id instead of the written library: `docs/V9.md`, `RUNBOOK.md`
section 1.4) on a Colab **TPU v6e-1** runtime:
downloads, tokenisation, the start-up benchmark and canary, about 19 hours of
training, post-training, the proxy evaluation and the export, inside one 24-hour budget of wall
time. Read `RUNBOOK.md` first (accounts, what the first 40 minutes must show, the fallback
ladder, what each failure means). Nothing in this repository has been measured on a TPU
(`docs/TPU_NOTES.md`).

Opened from the link in `README.md`, the notebook needs no edits to start: cell 1 already names
this repository and branch, and the notebook's metadata names a **v6e-1 TPU** runtime. Check
Runtime > Change runtime type before you connect all the same (cell 2 stops on any other device).

Order: run cells 1, 1b and 2 to 5, then **cell 6 to start a new run** or **cell 8 to continue
one** (after a disconnect, or on a recycled machine: never cell 6 then). Cell 7 shows progress
(run it as often as you like), the Drive mirror included. Cell 9 stops the run only after you set
`STOP_THE_RUN = True` in it, so Runtime > Run all never stops a run. Cells 10 and 11 (chat, blind
arena) are for after the run.

The code and every data file stay on the local disk of the runtime (about 72 GB for the default
plan); Drive (`USE_DRIVE`) receives only the run's records, logs, the newest checkpoint and the
export (`RUNBOOK.md` section 1.1 says how much space that needs).

The kernel of this notebook never imports JAX before the run: whichever process imports it first
on the TPU host owns the TPU.

In [ ]:
# 1. parameters: edit, then run the cells in order ------------------------------------------------
REPO_URL = "https://github.com/jdyar423-gif/TuringAward2027"   # git URL of this repository (a private one: CODE_COPY, RUNBOOK.md 1.1)
BRANCH = "claude/modest-carson-yi2d9g"   # the branch that holds this code ("" = the default branch)
CODE_COPY = ""                     # instead of cloning: a copy of the repository (for example on Drive),
                                   # copied to REPO_DIR without data_cache/ and runs/
REPO_DIR = "/content/TuringAward2027"   # the code the run uses: on the local disk, never on Drive
RUN_DIR = "/content/runs/v9"       # on the local disk; Drive only mirrors it
USE_DRIVE = True                   # mirror the run's records, logs, newest checkpoint and export to Drive
DRIVE_DIR = "/content/drive/MyDrive/bes_runs/v9"
CONFIG = "configs/v9.py"
BUDGET_MINUTES = None              # None: the configuration's 24 hours
ALLOW_CPU = False                  # True only for a rehearsal on a CPU runtime
OVERRIDES = ""                     # "section.field=value ..." (no spaces inside a value): the
                                   # --override options of a new run (RUNBOOK.md section 2.3)

In [ ]:
# 1b. helpers: run this cell, nothing to edit (no JAX in this kernel: it would take the TPU) -----
import json, os, shutil, signal, subprocess, sys, time

PID_FILE = RUN_DIR.rstrip("/") + ".pid"
LOG = RUN_DIR.rstrip("/") + ".log"
MIRROR_PID = RUN_DIR.rstrip("/") + ".mirror.pid"
MIRROR_STATUS = RUN_DIR.rstrip("/") + ".mirror.json"   # written by the mirror after every pass
MIRROR_PERIOD_S = 600             # one mirror pass every 10 minutes ...
MIRROR_CKPT_S = 1500              # ... and a new checkpoint copied to Drive at most every 25 minutes


def pid_alive(path=PID_FILE):
    """True when the process whose pid is in ``path`` is running."""
    try:
        pid = int(open(path).read().strip())
    except (OSError, ValueError):
        return False
    try:
        os.kill(pid, 0)
    except OSError:
        return False
    try:                                   # a zombie (finished, not reaped) is not running
        return open(f"/proc/{pid}/stat").read().split()[2] != "Z"
    except OSError:
        return True


def sh(cmd, env=None, cwd=None):
    """Run a command, print the tail of its output in this cell, raise with it on failure
    (a child's own output does not reach a notebook cell)."""
    p = subprocess.run(list(map(str, cmd)), capture_output=True, text=True, env=env, cwd=cwd)
    out = (p.stdout + p.stderr).strip()
    if p.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, cmd))} failed with exit status {p.returncode}:\n{out[-4000:]}")
    if out:
        print(out[-1500:])
    return p


def start_background(argv, log=LOG, pid_file=PID_FILE, env=None):
    """Start ``argv`` in its own session (it survives a kernel restart), output appended to ``log``."""
    os.makedirs(os.path.dirname(log) or ".", exist_ok=True)
    with open(log, "ab") as f:
        f.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} {' '.join(argv)}\n".encode())
        proc = subprocess.Popen(argv, stdout=f, stderr=subprocess.STDOUT, cwd=REPO_DIR,
                                env=dict(os.environ, PYTHONUNBUFFERED="1", **(env or {})),
                                start_new_session=True)
    with open(pid_file, "w") as f:
        f.write(str(proc.pid))
    return proc.pid


# Mirror of the run to Google Drive: the run's records (manifest, stage and training ledgers,
# the loader journal, reports, samples), its logs, the export and only the newest complete
# checkpoint of each checkpoint directory. No data: no downloads, token shards, library, models
# or compilation cache (a recycled machine fetches and builds them again). It never stops on an
# error; every pass is recorded in MIRROR_STATUS (cell 7 shows it).
MIRROR_CODE = r"""
import json, os, shutil, sys, time

# what is mirrored of a run directory: its records, logs, reports and the export (not state/
# and background.json: they describe this machine's background tasks, and a new machine
# runs those again) ...
ROOT_FILES = ("manifest.json", "stages.json", "stage_table.json")
FILES = ("train/ledger.json", "train/run.json", "train/train_log.jsonl", "train/loader_journal.jsonl",
         "benchmark/results.jsonl")
TREES = ("logs", "health", "post", "final")
# ... and the newest complete checkpoint of each checkpoint directory, nothing else
CKPT_ROOTS = ("train/ckpt", "train/ckpt_params", "post/ckpt")


def _skip(name):
    return (name.startswith((".tmp-", ".trash-", ".old-", ".part-")) or name.endswith((".part", ".lock"))
            or name == "__pycache__")


def _read(path):
    try:
        with open(path) as f:
            return json.load(f)
    except (OSError, ValueError):
        return None


def _copy(s, d):
    # one file, when its size or time changed, through a .part file
    st = os.stat(s)
    if os.path.exists(d):
        dt = os.stat(d)
        if dt.st_size == st.st_size and dt.st_mtime >= st.st_mtime:
            return 0
    os.makedirs(os.path.dirname(d) or ".", exist_ok=True)
    shutil.copy2(s, d + ".part")
    os.replace(d + ".part", d)
    return 1


def _files(src, top):
    # files of src/top (a file or a tree); checkpoint directories and temporaries left out
    path = os.path.join(src, top)
    if os.path.isfile(path):
        return [top]
    out = []
    for root, dirs, files in os.walk(path):
        rel = os.path.relpath(root, src)
        dirs[:] = sorted(d for d in dirs if not _skip(d) and os.path.join(rel, d) not in CKPT_ROOTS)
        out += [os.path.join(rel, f) for f in sorted(files) if not _skip(f)]
    return out


def _ckpts(root):
    try:
        return sorted(n for n in os.listdir(root) if n.startswith("ckpt-") and os.path.isdir(os.path.join(root, n)))
    except OSError:
        return []


def _same(a, b):
    # directory b holds every file of directory a with the same size
    try:
        return all(os.path.getsize(os.path.join(b, f)) == os.path.getsize(os.path.join(a, f))
                   for f in os.listdir(a))
    except OSError:
        return False


def sync(src, dst, st, ckpt_every_s=0.0, now=None):
    # One pass of a run directory into st (errors counted, never raised). Records, logs, reports
    # and the export are copied when they changed. Per checkpoint directory the newest local
    # checkpoint is copied into a .part- directory, renamed when complete, and only then are the
    # other checkpoints on Drive removed: Drive always holds one complete checkpoint. While
    # training runs a new checkpoint is copied at most every ckpt_every_s seconds, once the stage
    # ledger says training is over at once. A Drive directory that holds another run (another
    # manifest) is not touched.
    now = time.time() if now is None else now
    st.setdefault("checkpoints", {})
    st.setdefault("ckpt_time", {})
    if not os.path.isdir(src):
        return
    mine, theirs = _read(os.path.join(src, "manifest.json")), _read(os.path.join(dst, "manifest.json"))
    if mine and theirs and mine.get("created") != theirs.get("created"):
        raise RuntimeError(f"{dst} holds another run (created {theirs.get('created')}): not mirrored")
    os.makedirs(dst, exist_ok=True)
    for rel in [r for top in ROOT_FILES + FILES + TREES for r in _files(src, top)]:
        try:
            st["copied"] += _copy(os.path.join(src, rel), os.path.join(dst, rel))
        except OSError as e:
            st["errors"] += 1
            st["error"] = f"{rel}: {e}"
    train = ((_read(os.path.join(src, "stages.json")) or {}).get("stages") or {}).get("train") or {}
    training = train.get("status") == "started"
    for rel in CKPT_ROOTS:
        here = _ckpts(os.path.join(src, rel))
        if not here:
            continue
        new, there = here[-1], os.path.join(dst, rel)
        try:
            have = _ckpts(there)
            if new not in have or not _same(os.path.join(src, rel, new), os.path.join(there, new)):
                if have and training and now - st["ckpt_time"].get(rel, 0.0) < ckpt_every_s:
                    st["checkpoints"][rel] = have[-1]          # an older complete one stays
                    continue
                part = os.path.join(there, ".part-" + new)
                shutil.rmtree(part, ignore_errors=True)
                shutil.copytree(os.path.join(src, rel, new), part)
                if os.path.isdir(os.path.join(there, new)):
                    shutil.rmtree(os.path.join(there, new))
                os.replace(part, os.path.join(there, new))
                st["ckpt_time"][rel] = now
                st["copied"] += 1
            for name in os.listdir(there):                     # only now: the others go
                if name != new and (name.startswith("ckpt-") or name.startswith(".part-")):
                    shutil.rmtree(os.path.join(there, name), ignore_errors=True)
            st["checkpoints"][rel] = new
        except OSError as e:
            st["errors"] += 1
            st["error"] = f"{rel}/{new}: {e}"


def mirror_pass(pairs, status, ckpt_every_s=0.0, now=None):
    # One pass over every src=>dst pair (a run directory or a single file); updates status.
    now = time.time() if now is None else now
    st = {"errors": 0, "error": None, "copied": 0,
          "checkpoints": status.get("checkpoints", {}), "ckpt_time": status.get("ckpt_time", {})}
    for src, dst in pairs:
        try:
            if os.path.isdir(src):
                sync(src, dst, st, ckpt_every_s, now)
            elif os.path.isfile(src):
                st["copied"] += _copy(src, dst)
        except Exception as e:  # noqa: BLE001 - the mirror never dies
            st["errors"] += 1
            st["error"] = f"{src}: {type(e).__name__}: {e}"
    status.update(passes=status.get("passes", 0) + 1, last_pass=now, errors_last_pass=st["errors"],
                  errors_total=status.get("errors_total", 0) + st["errors"], copied_last_pass=st["copied"],
                  checkpoints=st["checkpoints"], ckpt_time=st["ckpt_time"])
    if st["error"]:
        status.update(last_error=st["error"], last_error_time=now)
    if not st["errors"]:
        status["last_good_pass"] = now
    try:
        status["drive_free_gb"] = round(shutil.disk_usage(pairs[0][1].rsplit("/", 1)[0] or "/").free / 1e9, 1)
    except (OSError, IndexError):
        status["drive_free_gb"] = None
    return status


if __name__ == "__main__":
    period, ckpt_every, status_path = float(sys.argv[1]), float(sys.argv[2]), sys.argv[3]
    pairs = [p.split("=>") for p in sys.argv[4:]]
    status = _read(status_path) or {}
    status.update(started=time.time(), period_s=period, ckpt_every_s=ckpt_every, pid=os.getpid())
    while True:
        try:
            mirror_pass(pairs, status, ckpt_every)
            with open(status_path + ".part", "w") as f:
                json.dump(status, f)
            os.replace(status_path + ".part", status_path)
            print(time.strftime("%H:%M:%S"), "mirrored", status.get("copied_last_pass"), "files,",
                  status.get("errors_last_pass"), "errors", flush=True)
        except Exception as e:  # noqa: BLE001
            print("mirror:", repr(e), flush=True)
        time.sleep(period)
"""


def start_mirror(period_s=MIRROR_PERIOD_S):
    """Mirror the run (records, logs, newest checkpoint, export) and its log to Drive (USE_DRIVE)."""
    if not USE_DRIVE:
        print("no Drive mirror (USE_DRIVE = False)")
        return
    if pid_alive(MIRROR_PID):
        print("the Drive mirror is already running")
        return
    code = RUN_DIR.rstrip("/") + ".mirror.py"
    with open(code, "w") as f:
        f.write(MIRROR_CODE)
    pairs = [f"{RUN_DIR}=>{DRIVE_DIR}", f"{LOG}=>{DRIVE_DIR.rstrip('/')}.log"]
    pid = start_background([sys.executable, code, str(period_s), str(MIRROR_CKPT_S), MIRROR_STATUS] + pairs,
                           log=RUN_DIR.rstrip("/") + ".mirror.log", pid_file=MIRROR_PID)
    print(f"Drive mirror started (pid {pid}) every {period_s / 60:.0f} min -> {DRIVE_DIR}")


def restore_from_drive():
    """After a recycled machine: copy the mirrored run back (records, logs, the newest checkpoint,
    the export). The data is downloaded, tokenised and built again by the resumed run."""
    if os.path.isdir(DRIVE_DIR) and not os.path.exists(os.path.join(RUN_DIR, "manifest.json")):
        print(f"restoring {DRIVE_DIR} -> {RUN_DIR}")
        shutil.copytree(DRIVE_DIR, RUN_DIR, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("*.part", ".part-*", ".tmp-*"))
    if os.path.isfile(DRIVE_DIR.rstrip("/") + ".log") and not os.path.exists(LOG):
        shutil.copy2(DRIVE_DIR.rstrip("/") + ".log", LOG)


def _read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except (OSError, ValueError):
        return None


def _secs(x):
    return "-" if not isinstance(x, (int, float)) else f"{x / 3600:.2f} h" if x >= 3600 else f"{x / 60:.1f} min" if x >= 60 else f"{x:.0f} s"


def mirror_report(now=None):
    """Lines about the Drive mirror from MIRROR_STATUS, with a warning when it is stale."""
    if not USE_DRIVE:
        return ["Drive mirror: off (USE_DRIVE = False)"]
    now = time.time() if now is None else now
    ms = _read_json(MIRROR_STATUS)
    alive = pid_alive(MIRROR_PID)
    if not ms:
        return [f"Drive mirror: {'running, no pass recorded yet' if alive else 'NOT running (cell 6 or 8 starts it)'}"]
    good = ms.get("last_good_pass")
    out = [f"Drive mirror {'running' if alive else 'NOT running'}: {ms.get('passes', 0)} passes, last good pass "
           f"{_secs(now - good) + ' ago' if good else 'never'}, errors {ms.get('errors_total', 0)} "
           f"(last pass {ms.get('errors_last_pass', 0)}), Drive free {ms.get('drive_free_gb')} GB"]
    if ms.get("last_error"):
        out.append(f"  last error ({_secs(now - ms.get('last_error_time', now))} ago): {str(ms['last_error'])[:200]}")
    out.append(f"  newest checkpoint on Drive: {ms.get('checkpoints') or 'none yet'}")
    if not alive or not good or now - good > 2 * MIRROR_PERIOD_S + 60:
        out.append("  WARNING: the Drive copy is stale or the mirror stopped: a recycled runtime would "
                   "resume from it (RUNBOOK.md section 4)")
    return out


def show_progress(run_dir=RUN_DIR, tail=12):
    """Stage table, benchmark and canary, the last training steps, background tasks, the Drive
    mirror, the log."""
    print(f"run {'RUNNING' if pid_alive() else 'NOT running'} ({run_dir})")
    man = _read_json(os.path.join(run_dir, "manifest.json")) or {}
    st = _read_json(os.path.join(run_dir, "stages.json")) or {}
    b = man.get("budget") or {}
    if b:
        left = b["deadline"] - time.time()
        print(f"budget {_secs(b['budget_s'])}, elapsed {_secs(time.time() - b['t0'])}, left {_secs(max(0.0, left))}")
    print("\n stage        status     planned     used  note")
    for name in ("preflight", "background", "benchmark", "train", "sample", "critic", "prefopt",
                 "select", "judge", "export"):
        r = (st.get("stages") or {}).get(name) or {}
        print(f" {name:12s} {r.get('status', '-'):9s} {_secs(r.get('planned_s')):>9s} {_secs(r.get('used_s')):>8s}  {(r.get('reason') or '')[:70]}")
    bench = man.get("benchmark") or {}
    if bench:
        print(f"\nbenchmark: plan {bench.get('plan_key')}, expected {bench.get('expected_tps', 0):,.0f} tok/s, horizon {bench.get('horizon_tokens', 0):.3e} tokens")
        for m in bench.get("measurements") or []:
            p = m.get("plan") or {}
            key = f"{p.get('attn_impl')}-remat{int(bool(p.get('remat')))}-ms{p.get('micro_seqs')}-{p.get('table_grad')}"
            peak = (m.get("peak_bytes") or 0) / 2 ** 30
            print(f"  {key:28s} T={m.get('seq_len')}: {'ok ' if m.get('ok') else 'NO '} compile {m.get('compile_s') or 0:6.1f} s, step {m.get('step_s') or 0:6.3f} s, {m.get('tokens_per_s') or 0:10,.0f} tok/s, peak {peak:5.2f} GiB {('' if m.get('ok') else m.get('reason', ''))[:60]}")
        can = bench.get("canary") or {}
        lad = bench.get("ladder") or {}
        print("  canary: " + ", ".join(f"{k} {(can.get(k) or {}).get('ok')}" for k in ("parity", "determinism", "safe", "full")))
        print(f"  fallback ladder rung {lad.get('rung')}" + (f": {', '.join(lad.get('groups') or [])} off" if lad.get("groups") else " (full configuration)"))
        h = bench.get("host") or {}
        print(f"  host loader {h.get('rows_per_s', 0):,.0f} rows/s with {h.get('workers')} workers, device needs {h.get('needed_rows_per_s', 0):,.0f}; workers for training {h.get('loader_workers')}")
    led = _read_json(os.path.join(run_dir, "train", "ledger.json")) or {}
    if led:
        s = led.get("seconds") or {}
        print(f"\ntraining ledger: {led.get('tokens', 0):,} tokens, {led.get('steps', 0):,} steps, train {_secs(s.get('train'))}, lost {_secs(s.get('lost'))}, restarts {led.get('restarts')}, rollbacks {led.get('rollbacks')}")
    path = os.path.join(run_dir, "train", "train_log.jsonl")
    if os.path.exists(path):
        steps, events = [], []
        with open(path) as f:
            for line in f:
                try:
                    r = json.loads(line)
                except ValueError:
                    continue
                (steps if r.get("event") == "step" else events).append(r)
        for r in steps[-5:]:
            print(f"  step {r['step']:7d}  p {r['p']:.4f}  T {r.get('seq_len')}  loss {r['loss']:.4f}  ema {r.get('loss_ema', float('nan')):.4f}  {r.get('tok_per_s', 0):10,.0f} tok/s  lr x{r.get('lr', 0):.3g}  {r.get('action', '')}")
        for r in events[-4:]:
            print(f"  event {r.get('event')}: " + ", ".join(f"{k}={r[k]}" for k in ("step", "reason", "path", "p") if k in r))
    bg = (_read_json(os.path.join(run_dir, "background.json")) or {}).get("tasks") or {}
    if bg:
        print("\nbackground: " + ", ".join(f"{k} {v.get('state')}" + (f" ({v.get('error')[:40]})" if v.get("error") else "") for k, v in bg.items()))
    hl = _read_json(os.path.join(run_dir, "health", "latest.json"))
    if hl:
        print(f"side-car (step {(hl.get('checkpoint') or {}).get('step')}): probe loss {hl.get('probe_loss')}")
    print("\n" + "\n".join(mirror_report()))
    if os.path.exists(LOG):
        with open(LOG, errors="replace") as f:
            lines = f.readlines()[-tail:]
        print("\n--- log tail ---\n" + "".join(lines))

In [ ]:
# 2. runtime check (in a separate process: this kernel must not take the TPU) -----------------------
import json, os, shutil, subprocess, sys

probe = ("import json, jax, jaxlib; d = jax.devices(); print(json.dumps({'jax': jax.__version__, "
         "'jaxlib': jaxlib.__version__, 'backend': jax.default_backend(), 'kinds': [x.device_kind for x in d]}))")
out = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, timeout=900)
if out.returncode != 0:
    raise RuntimeError("JAX does not start on this runtime:\n" + out.stderr[-3000:])
JAX_INFO = json.loads(out.stdout.strip().splitlines()[-1])
print(JAX_INFO)
if JAX_INFO["backend"] != "tpu" and not ALLOW_CPU:
    raise RuntimeError(f"no TPU (backend {JAX_INFO['backend']}): choose Runtime > Change runtime type > TPU v6e-1")
if JAX_INFO["backend"] == "tpu" and not any("v6" in k for k in JAX_INFO["kinds"]):
    raise RuntimeError(f"this is not a TPU v6e ({JAX_INFO['kinds']}): the memory plan and the time budget assume one v6e chip")
if len(JAX_INFO["kinds"]) != 1:
    print(f"WARNING: {len(JAX_INFO['kinds'])} devices; the run uses device 0 only")
if JAX_INFO["jax"] != "0.11.1":
    print(f"WARNING: the runtime has jax {JAX_INFO['jax']}; the code was compile-checked with 0.11.1. "
          "It is not upgraded or downgraded here; the run's preflight records the versions.")
mem_gb = 0.0
for line in open("/proc/meminfo"):
    if line.startswith("MemTotal:"):
        mem_gb = int(line.split()[1]) / 1e6
du = shutil.disk_usage("/content")
print(f"{os.cpu_count()} host cores, {mem_gb:.0f} GB RAM, {du.free / 1e9:.0f} GB free disk under /content")
if du.free / 1e9 < 80:
    print("WARNING: the default plan needs about 72 GB of local disk (token shards plus the raw files not "
          "yet tokenised: a raw file is deleted once its shard is committed); the preflight stops the run if "
          "the estimate does not fit (RUNBOOK.md: lower the download caps)")

In [ ]:
# 3. the code, on the local disk: clone it, or copy CODE_COPY (for example a copy on Drive) ----------
# Every data path of the run (downloads, token shards, the library, the opponents' models, the proxy
# outputs) is relative to the code's directory, so the code runs from the local disk: data never
# goes to Drive. Drive only receives the mirror (cell 6 / 8) and the export (cell 12).
def _on_drive(path):
    p = os.path.realpath(path) if path else ""
    return p == "/content/drive" or p.startswith("/content/drive/")


if _on_drive(REPO_DIR):
    raise RuntimeError(f"REPO_DIR = {REPO_DIR} is on Google Drive: keep REPO_DIR on the local disk and give "
                       "the copy on Drive as CODE_COPY (cell 1); it is copied to REPO_DIR without data")
if USE_DRIVE or _on_drive(CODE_COPY):
    from google.colab import drive
    drive.mount("/content/drive")
if CODE_COPY:
    if not os.path.isdir(os.path.join(CODE_COPY, "bes")):
        raise RuntimeError(f"CODE_COPY = {CODE_COPY} does not hold the repository (no bes/ in it)")
    shutil.copytree(CODE_COPY, REPO_DIR, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("data_cache", "runs", "__pycache__", ".pytest_cache"))
    print(f"copied the code from {CODE_COPY} to {REPO_DIR} (without data_cache/ and runs/)")
elif os.path.isdir(os.path.join(REPO_DIR, "bes")):
    print(f"using the code in {REPO_DIR}")
else:
    if not REPO_URL:
        raise RuntimeError("set REPO_URL (and BRANCH) or CODE_COPY in cell 1")
    sh(["git", "clone", "--depth", "1"] + (["--branch", BRANCH] if BRANCH else []) + [REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--format=%H %ad %s"], capture_output=True, text=True).stdout)

In [ ]:
# 4. dependencies (requirements-colab.txt) without touching the preinstalled jax ------------------
# llama-cpp-python is built from source: this cell can take 5 to 15 minutes.
pin_code = ("import importlib.metadata as m\n"
            "for n in ('jax', 'jaxlib', 'libtpu', 'libtpu-nightly'):\n"
            "    try:\n        print(n + '==' + m.version(n))\n"
            "    except m.PackageNotFoundError:\n        pass\n")
held = subprocess.run([sys.executable, "-c", pin_code], capture_output=True, text=True).stdout.split()
with open("/content/constraints-jax.txt", "w") as f:
    f.write("\n".join(held) + "\n")
print("held at their installed versions:", held)
env = dict(os.environ, CMAKE_BUILD_PARALLEL_LEVEL=str(os.cpu_count() or 4))
sh([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt",
    "-c", "/content/constraints-jax.txt"], env=env)
after = subprocess.run([sys.executable, "-c", "import jax, jaxlib; print(jax.__version__, jaxlib.__version__)"],
                       capture_output=True, text=True).stdout.split()
if after != [JAX_INFO["jax"], JAX_INFO["jaxlib"]]:
    raise RuntimeError(f"jax changed during the install ({JAX_INFO['jax']} -> {after}): restart the runtime and investigate")
check = subprocess.run([sys.executable, "-c", "import llama_cpp, optax, tiktoken, rustbpe, pyarrow, huggingface_hub, regex; print('imports ok')"],
                       capture_output=True, text=True)
if check.returncode != 0:
    raise RuntimeError("the dependencies do not import:\n" + check.stderr[-2000:])
print(check.stdout.strip())

In [ ]:
# 5. Hugging Face token (kept in this kernel's memory only, never written to the notebook) ---------
from getpass import getpass

token = ""
try:
    from google.colab import userdata          # a Colab secret named HF_TOKEN, if you granted access
    token = userdata.get("HF_TOKEN") or ""
except Exception:
    pass
if not token:
    token = getpass("Hugging Face token (Enter to skip): ").strip()
if token:
    os.environ["HF_TOKEN"] = token
del token
print("HF_TOKEN", "set" if os.environ.get("HF_TOKEN") else "not set: public files only, lower rate limits")

In [ ]:
# 6. start the 24-hour run in the background ---------------------------------------------------------------
# A new run only: on a recycled machine, or after a disconnect, use cell 8 instead (it continues the
# run with its budget and checkpoints; starting a new one here would replace the run on Drive).
os.makedirs(os.path.dirname(RUN_DIR), exist_ok=True)
if pid_alive():
    raise RuntimeError("a run started from this notebook is still running (cell 7 shows it)")
if os.path.exists(os.path.join(RUN_DIR, "manifest.json")):
    raise RuntimeError(f"{RUN_DIR} already holds a run: continue it with cell 8, or choose another RUN_DIR")
if USE_DRIVE and os.path.exists(os.path.join(DRIVE_DIR, "manifest.json")):
    raise RuntimeError(f"Drive already holds a run in {DRIVE_DIR}: continue it with cell 8 (it copies the run "
                       "back from Drive first), or set another DRIVE_DIR and RUN_DIR in cell 1 for a new run")
argv = [sys.executable, "scripts/run_24h.py", "--config", CONFIG, "--out", RUN_DIR]
if BUDGET_MINUTES:
    argv += ["--budget-minutes", str(BUDGET_MINUTES)]
for override in OVERRIDES.split():
    argv += ["--override", override]
if ALLOW_CPU:
    argv += ["--allow-cpu"]
pid = start_background(argv)
print(f"started pid {pid}: {' '.join(argv)}\nlog: {LOG}")
start_mirror()

In [ ]:
# 7. progress (run this cell whenever you like) -----------------------------------------------------
show_progress()

In [ ]:
# 8. continue the run (after a disconnect, a crash, a stop, or on a recycled machine) ----------------
# On a recycled machine run cells 1, 1b and 2 to 5 first (code, dependencies, token), then this cell
# (not cell 6): it copies the mirrored run (records, logs, newest checkpoint) back from Drive and
# starts scripts/run_24h.py --resume (same budget: the downtime counts against it). The data is
# downloaded, tokenised and built again by the run, in the background, as at the start.
if pid_alive():
    print("the run is still running: nothing to do (cell 7 shows its progress)")
else:
    if not os.path.exists(os.path.join(RUN_DIR, "manifest.json")) and USE_DRIVE:
        restore_from_drive()
    if not os.path.exists(os.path.join(RUN_DIR, "manifest.json")):
        raise RuntimeError(f"no run to continue in {RUN_DIR} (and none mirrored in {DRIVE_DIR})")
    argv = [sys.executable, "scripts/run_24h.py", "--out", RUN_DIR, "--resume"]
    if ALLOW_CPU:
        argv += ["--allow-cpu"]
    pid = start_background(argv)
    print(f"resumed: pid {pid}; log {LOG}")
    start_mirror()

In [ ]:
# 9. stop the run cleanly (only if you must): set STOP_THE_RUN = True here and run this cell -------
# The trainer saves a checkpoint and the run exits; cell 8 continues it. "Run all" never stops a run:
# the flag is False unless you change it.
STOP_THE_RUN = False
if not STOP_THE_RUN:
    print("not stopping anything: set STOP_THE_RUN = True in this cell to send SIGTERM to the run")
elif pid_alive():
    os.kill(int(open(PID_FILE).read()), signal.SIGTERM)
    print("SIGTERM sent; the run stops after its current step and writes a final checkpoint")
else:
    print("the run is not running")

## After the run

`RUN_DIR/final/` holds the export (`MODEL_CARD.md` first: the held-out check of the 16-bit
weights, the accounting, the read-set audit, the stage table, the evaluation results that exist).
The chat and the arena below use it. If the run is still training they run on the host CPU (the
run owns the TPU); otherwise this kernel takes the TPU, so do not start another run from this
runtime afterwards without restarting it.

In [ ]:
# 10. chat with the exported model (type /quit to leave) ------------------------------------------
import importlib.util

EXPORT = os.path.join(RUN_DIR, "final")
if pid_alive():
    os.environ["JAX_PLATFORMS"] = "cpu"
    print("the run still owns the TPU: the chat runs on the CPU")
spec = importlib.util.spec_from_file_location("bes_chat", os.path.join(REPO_DIR, "scripts", "chat.py"))
bes_chat = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bes_chat)
bes_chat.main(["--export", EXPORT, "--show-diagnostics"])

In [ ]:
# 11. blind arena: our model against the opponents (q quits; votes are saved after each one) --------
import importlib.util

REVEAL = False                     # True: show which model wrote which reply after each vote
PROMPTS = None                     # None: type prompts; or "proxy:test", or a .txt / .jsonl file
args = ["--export", os.path.join(RUN_DIR, "final"), "--name", "final"]
if REVEAL:
    args.append("--reveal")
if PROMPTS:
    args += ["--prompts", PROMPTS]
if pid_alive():
    os.environ["JAX_PLATFORMS"] = "cpu"
spec = importlib.util.spec_from_file_location("bes_arena", os.path.join(REPO_DIR, "scripts", "arena.py"))
bes_arena = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bes_arena)
bes_arena.main(args)

In [ ]:
# 12. copy the export (and the arena votes) to Drive -----------------------------------------------
if USE_DRIVE:
    for src, dst in ((os.path.join(RUN_DIR, "final"), DRIVE_DIR + ".export"),
                     (os.path.join(REPO_DIR, "data_cache", "arena"), DRIVE_DIR + ".arena")):
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"copied {src} -> {dst}")
v = subprocess.run([sys.executable, "scripts/export.py", "--verify", os.path.join(RUN_DIR, "final")],
                   capture_output=True, text=True)
print(v.stdout + v.stderr)